# Day 2 前提条件の確認

Day 1で作成したリソースと、Dify環境の準備状況を確認します。

## 使用するモデル

Day 2では **`databricks-gpt-5-2`**（Foundation Model API）を使用します。

> **なぜGPT-5.2？**  
> DifyのOpenAI-API-compatibleプラグインは全リクエストに `user` パラメータを付与します。  
> GPT系モデルはこのパラメータを受け入れますが、Claude/Gemini系は拒否するため、  
> **プラグインの修正なしで動作するGPT-5.2を採用**しています。

In [ ]:
%run ./config

## Day 1 リソース確認
> Day 2はCode スキーマ（`ricoh_handson_code`、日本語データ）を使用します。

In [ ]:
# テーブル確認
for table in ["products", "customers", "orders", "support_tickets", "policies", "knowledge_base"]:
    try:
        count = spark.table(f"{catalog}.{schema_code}.{table}").count()
        print(f"✅ {schema_code}.{table}: {count}件")
    except Exception as e:
        print(f"❌ {schema_code}.{table}: 未作成 ({e})")

In [ ]:
# UC関数確認
functions = spark.sql(f"SHOW FUNCTIONS IN {catalog}.{schema_code}").collect()
print("✅ 登録済みUC関数:")
for f in functions:
    if not f.function.startswith("__"):
        print(f"  - {f.function}")

## Dify側の前提設定

### SSRF Proxy設定

Difyは外部通信をSSRF Proxyで制限しています。  
Databricksと接続するには、Dify管理者がSquid設定で以下のドメインを許可してください：

```
.cloud.databricks.com
.ai-gateway.cloud.databricks.com
```

この設定がないと、DifyからDatabricksへの通信がすべてブロックされます。

### OpenAI-API-compatibleプラグイン

Difyの **Plugins** → **Marketplace** から **OpenAI-API-compatible** プラグイン（v0.0.40以上）をインストールしてください。  
このプラグインを使ってDatabricksのLLMエンドポイントに接続します。

## Databricks PATトークンの生成
Dify連携に必要なPersonal Access Token（PAT）を生成します。
1. 右上のユーザーアイコン → **Settings**
2. **Developer** → **Access tokens**
3. **Generate new token** をクリック
4. トークンをコピーして安全に保管

In [ ]:
import os
# トークンの確認（環境変数から取得）
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host = spark.conf.get("spark.databricks.workspaceUrl")
print(f"✅ Workspace URL: https://{host}")
print(f"✅ トークン取得済み（先頭10文字）: {token[:10]}...")

## Day 2で使用する接続情報

In [ ]:
print("=" * 60)
print("Day 2 Dify連携用の接続情報")
print("=" * 60)
print(f"\n【Pattern ① LLM連携】")
print(f"  Base URL: https://{host}/serving-endpoints")
print(f"  Model: {NOCODE_LLM_ENDPOINT_NAME}")
print(f"\n【Pattern ② HTTP API連携】")
print(f"  SQL API: https://{host}/api/2.0/sql/statements")
print(f"\n【Pattern ③ MCP連携】")
print(f"  MCP URL: https://{host}/api/2.0/mcp/functions/{catalog}/{schema_code}")
print(f"\n【共通】")
print(f"  Authorization: Bearer <your_PAT_token>")